In [2]:
import pandas as pd
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(
    '../US_Accidents_March23.csv',
    usecols=['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng',
              'Temperature(F)', 'Humidity(%)', 'Visibility(mi)',
              'Wind_Speed(mph)', 'Precipitation(in)',
              'Weather_Condition', 'Sunrise_Sunset'],
    nrows=200000
)

print("Initial shape:", df.shape)


df['Severity_Binary'] = df['Severity'].apply(lambda x: 0 if x in [1, 2] else 1)
print("\nBinary severity distribution:")
print(df['Severity_Binary'].value_counts())
print(df['Severity_Binary'].value_counts(normalize=True).round(3))


df = df.drop(columns=['Severity'])


df = df.drop(columns=['Precipitation(in)'])


df = df.dropna(subset=['Weather_Condition', 'Sunrise_Sunset'])



for col in ['Wind_Speed(mph)', 'Visibility(mi)', 'Humidity(%)', 'Temperature(F)']:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df.isnull().sum())


df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['Hour'] = df['Start_Time'].dt.hour
df = df.drop(columns=['Start_Time'])  # no longer needed after extracting Hour


df['Sunrise_Sunset'] = df['Sunrise_Sunset'].map({'Day': 0, 'Night': 1})


top_weather = df['Weather_Condition'].value_counts().nlargest(10).index
df['Weather_Condition'] = df['Weather_Condition'].where(
    df['Weather_Condition'].isin(top_weather), 'Other'
)
df = pd.get_dummies(df, columns=['Weather_Condition'], drop_first=True)


scaler = StandardScaler()
numeric_cols = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)']
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("\nFinal shape after outlier removal:", df.shape)
print("\nFinal columns:", df.columns.tolist())
print("\nMissing values check (should all be 0):")
print(df.isnull().sum().sum())


df.to_csv('../data/processed/accidents_cleaned.csv', index=False)
print("\nSaved to data/processed/accidents_cleaned.csv")
df.head()

  Using cached scikit_learn-1.9.1-cp314-cp314-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp314-cp314-macosx_12_0_arm64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl (20.5 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Initial shape: (200000, 11)

Binary severity distribution:
Severity

,Start_Lat,Start_Lng,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Sunrise_Sunset,Severity_Binary,Hour,Weather_Condition_Fair,Weather_Condition_Haze,Weather_Condition_Light Rain,Weather_Condition_Light Snow,Weather_Condition_Mostly Cloudy,Weather_Condition_Other,Weather_Condition_Overcast,Weather_Condition_Partly Cloudy,Weather_Condition_Rain,Weather_Condition_Scattered Clouds
0,39.865147,-84.058723,-1.759378,1.259124,0.359699,-0.058481,1,1,5,False,False,True,False,False,False,False,False,False,False
1,39.928059,-82.831184,-1.698327,1.677430,0.359699,-0.058481,1,0,6,False,False,True,False,False,False,False,False,False,False
2,39.063148,-84.032608,-1.814324,1.677430,0.359699,-0.864060,1,0,6,False,False,False,False,False,False,True,False,False,False
13,39.790760,-84.241547,-1.814324,1.166167,0.359699,-0.461271,0,0,8,False,False,False,False,True,False,False,False,False,False
17,39.752174,-84.239952,-1.814324,1.166167,0.359699,-0.461271,0,0,9,False,False,False,False,True,False,False,False,False,False
